[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonasneves/colab-slm-playground/blob/main/trending_onnx_chatbot_colab.ipynb)

# Trending ONNX Chatbot

Fetches trending `text-generation` ONNX models from Hugging Face, lets you pick one, and runs a chat UI entirely inside Colab.

Uses [`optimum[onnxruntime]`](https://huggingface.co/docs/optimum/onnxruntime/overview) for loading — no manual session management.

## 1 — Install dependencies

In [ ]:
!pip install -q "optimum[onnxruntime]" transformers huggingface_hub ipywidgets numpy

## 2 — Fetch trending models

In [ ]:
from huggingface_hub import HfApi
from IPython.display import display, HTML

print("Fetching trending ONNX text-generation models...")

api = HfApi()
raw_models = list(api.list_models(
    library="onnx",
    pipeline_tag="text-generation",
    sort="trending_score",
    direction=-1,
    limit=30,
    full=True,
))

model_info = []
for m in raw_models:
    has_onnx = any(
        s.rfilename.endswith(".onnx")
        for s in (m.siblings or [])
    )
    if not has_onnx:
        continue
    model_info.append({
        "id": m.id,
        "downloads": m.downloads or 0,
        "likes": m.likes or 0,
    })

rows = "".join(
    f"<tr>"
    f"<td>{i+1}</td>"
    f"<td><code>{m['id']}</code></td>"
    f"<td>{m['downloads']:,}</td>"
    f"<td>{m['likes']:,}</td>"
    f"</tr>"
    for i, m in enumerate(model_info)
)

display(HTML(f"""
<style>
  .ort-table {{
    border-collapse: collapse;
    font-family: monospace;
    font-size: 13px;
    color-scheme: light dark;
  }}
  .ort-table td, .ort-table th {{
    padding: 5px 12px;
  }}
  .ort-table th {{
    text-align: left;
    border-bottom: 2px solid;
  }}
  .ort-table td:nth-child(3),
  .ort-table td:nth-child(4),
  .ort-table th:nth-child(3),
  .ort-table th:nth-child(4) {{
    text-align: right;
  }}
  @media (prefers-color-scheme: light) {{
    .ort-table th {{ background: #f0f0f0; border-color: #bbb; color: #111; }}
    .ort-table td {{ color: #111; }}
    .ort-table tr:hover td {{ background: #f7f7f7; }}
  }}
  @media (prefers-color-scheme: dark) {{
    .ort-table th {{ background: #2a2a2a; border-color: #555; color: #ddd; }}
    .ort-table td {{ color: #ccc; }}
    .ort-table tr:hover td {{ background: #1e1e1e; }}
  }}
</style>
<table class="ort-table">
  <thead>
    <tr>
      <th>#</th>
      <th>Model ID</th>
      <th>Downloads</th>
      <th>Likes</th>
    </tr>
  </thead>
  <tbody>{rows}</tbody>
</table>
"""))

print(f"Found {len(model_info)} models with ONNX files.")

## 3 — Select & load model

Pick a model from the dropdown, then click **Load Model**. The model is loaded when you click — not when the cell runs — so the selection is always captured correctly.

In [ ]:
import ipywidgets as widgets
from optimum.onnxruntime import ORTModelForCausalLM
from transformers import AutoTokenizer, PreTrainedTokenizerFast
from IPython.display import display

model = None
tokenizer = None

model_dropdown = widgets.Dropdown(
    options=[m['id'] for m in model_info],
    description="Model:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="620px"),
)

load_btn = widgets.Button(
    description="Load Model",
    button_style="primary",
    layout=widgets.Layout(width="140px", height="36px"),
)

status = widgets.Output()

_TOKENIZER_CLASS_ERR = ("does not exist", "not currently imported")

def _load_tokenizer(model_id):
    try:
        return AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    except Exception as e:
        if any(msg in str(e) for msg in _TOKENIZER_CLASS_ERR):
            print("Custom tokenizer class unavailable, loading tokenizer.json directly...")
            return PreTrainedTokenizerFast.from_pretrained(model_id)
        raise

_COMPAT_ERRORS = {
    "'list' object has no attribute 'keys'": (
        "Model has a non-standard ONNX config that ORTModelForCausalLM can't parse.\n"
        "If you're trying to load LiquidAI/LFM2.5-350M-ONNX, use lfm2_chatbot_colab.ipynb instead."
    ),
    "Could not find any ONNX": (
        "No standard ONNX file found. The model may use a custom layout incompatible with Optimum.\n"
        "Try a different model from the list."
    ),
}

def _friendly_error(e):
    msg = str(e)
    for pattern, hint in _COMPAT_ERRORS.items():
        if pattern in msg:
            return hint
    return f"Error: {msg}"

def on_load(b):
    global model, tokenizer
    selected_id = model_dropdown.value
    load_btn.disabled = True
    load_btn.description = "Loading..."
    with status:
        status.clear_output()
        print(f"Loading: {selected_id}")
        print("Downloading model — this may take a few minutes...")
        try:
            tokenizer = _load_tokenizer(selected_id)
            model = ORTModelForCausalLM.from_pretrained(selected_id, trust_remote_code=True)
            if tokenizer.pad_token_id is None:
                tokenizer.pad_token_id = tokenizer.eos_token_id
            print("Ready. Run the next cell to open the chat.")
            load_btn.description = "Loaded"
            load_btn.button_style = "success"
        except Exception as e:
            print(_friendly_error(e))
            load_btn.disabled = False
            load_btn.description = "Load Model"

load_btn.on_click(on_load)

display(model_dropdown, load_btn, status)

## 4 — Launch Chat UI

In [ ]:
import json
from google.colab import output as colab_output
from IPython.display import display, HTML
from IPython.display import JSON as IPJSON

assert model is not None, "Load a model first (cell above)."

MAX_NEW_TOKENS = 512


def _build_prompt(messages: list) -> str:
    if tokenizer.chat_template:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    parts = []
    for m in messages:
        role = "User" if m["role"] == "user" else "Assistant"
        parts.append(f"{role}: {m['content']}")
    return "\n".join(parts) + "\nAssistant:"


def generate(messages: list) -> str:
    prompt = _build_prompt(messages)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    input_len = inputs["input_ids"].shape[1]

    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.pad_token_id,
    )

    new_tokens = output_ids[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def _chat_callback(messages_json):
    try:
        messages = json.loads(messages_json)
        reply = generate(messages)
        return IPJSON({"reply": reply})
    except Exception as e:
        return IPJSON({"error": str(e)})


colab_output.register_callback("notebook.chat", _chat_callback)

short_name = model_dropdown.value.split("/")[-1]

CHAT_HTML = f"""
<div id="ort-chat" style="
  font-family: 'Segoe UI', system-ui, -apple-system, sans-serif;
  max-width: 640px;
  border: 1px solid #d0d0d0;
  border-radius: 12px;
  overflow: hidden;
  background: #fafafa;
">
  <div style="
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
    color: white;
    padding: 14px 18px;
    font-size: 15px;
    font-weight: 600;
    display: flex;
    align-items: center;
    gap: 8px;
  ">
    <span style="font-size: 20px;">&#x1F916;</span>
    {short_name}
    &nbsp;<span style="font-weight:400; opacity:0.7; font-size:12px;">ONNX</span>
  </div>

  <div id="ort-messages" style="
    height: 380px;
    overflow-y: auto;
    padding: 16px;
    display: flex;
    flex-direction: column;
    gap: 10px;
  "></div>

  <div style="
    display: flex;
    border-top: 1px solid #e0e0e0;
    background: white;
    padding: 8px;
    gap: 8px;
  ">
    <input id="ort-input" type="text" placeholder="Type a message..."
      style="
        flex: 1;
        border: 1px solid #d0d0d0;
        border-radius: 8px;
        padding: 10px 14px;
        font-size: 14px;
        outline: none;
      "
    />
    <button id="ort-send" onclick="ortSend()" style="
      background: #1a1a2e;
      color: white;
      border: none;
      border-radius: 8px;
      padding: 10px 18px;
      font-size: 14px;
      cursor: pointer;
      font-weight: 600;
    ">Send</button>
  </div>
</div>

<script>
(function() {{
  const messagesDiv = document.getElementById("ort-messages");
  const inputEl    = document.getElementById("ort-input");
  const sendBtn    = document.getElementById("ort-send");
  let history = [];

  function addBubble(role, text) {{
    const isUser = role === "user";
    const bubble = document.createElement("div");
    bubble.style.cssText = `
      max-width: 82%;
      padding: 10px 14px;
      border-radius: 12px;
      font-size: 14px;
      line-height: 1.5;
      word-wrap: break-word;
      white-space: pre-wrap;
      align-self: ${{isUser ? "flex-end" : "flex-start"}};
      background: ${{isUser ? "#1a1a2e" : "#e8e8e8"}};
      color: ${{isUser ? "#fff" : "#1a1a1a"}};
    `;
    bubble.textContent = text;
    messagesDiv.appendChild(bubble);
    messagesDiv.scrollTop = messagesDiv.scrollHeight;
    return bubble;
  }}

  function extractPayload(result) {{
    if (result && result.data) {{
      if (result.data["application/json"]) {{
        const v = result.data["application/json"];
        return (typeof v === "string") ? JSON.parse(v) : v;
      }}
      if (result.data["text/plain"]) {{
        let t = result.data["text/plain"];
        t = t.replace(/^['"]/,"").replace(/['"]$/,"");
        t = t.replace(/\\'/g, "'").replace(/\\"/g, '"');
        return JSON.parse(t);
      }}
    }}
    if (result && result.reply) return result;
    throw new Error("Could not parse kernel response: " + JSON.stringify(result).substring(0, 200));
  }}

  window.ortSend = async function() {{
    const text = inputEl.value.trim();
    if (!text) return;
    inputEl.value = "";
    inputEl.disabled = true;
    sendBtn.disabled = true;
    sendBtn.textContent = "...";

    addBubble("user", text);
    history.push({{role: "user", content: text}});
    const thinking = addBubble("assistant", "Thinking...");

    try {{
      const result = await google.colab.kernel.invokeFunction(
        "notebook.chat",
        [JSON.stringify(history)],
        {{}}
      );
      const data = extractPayload(result);
      if (data.error) throw new Error(data.error);
      thinking.textContent = data.reply;
      history.push({{role: "assistant", content: data.reply}});
    }} catch (err) {{
      thinking.textContent = "Error: " + err.message;
      thinking.style.background = "#ffe0e0";
      thinking.style.color = "#900";
      history.pop();
    }}

    inputEl.disabled = false;
    sendBtn.disabled = false;
    sendBtn.textContent = "Send";
    inputEl.focus();
  }};

  inputEl.addEventListener("keydown", (e) => {{
    if (e.key === "Enter" && !e.shiftKey) {{ e.preventDefault(); ortSend(); }}
  }});
}})();
</script>
"""

display(HTML(CHAT_HTML))
print("Chat ready. Each response may take 10–60 s depending on model size.")

---
### Notes

**What is ONNX and why run it on CPU?**
ONNX (Open Neural Network Exchange) is a portable model format that separates the model weights from any specific ML framework. Once a model is exported to ONNX, it can run via ONNX Runtime without PyTorch installed — which means lighter dependencies, faster cold starts, and CPU inference that's often faster than PyTorch on CPU thanks to runtime-level graph optimizations.

**What is Optimum?**
[Optimum](https://huggingface.co/docs/optimum) is Hugging Face's library for exporting and running transformer models on accelerated runtimes. `ORTModelForCausalLM` wraps an ONNX session to expose the same `.generate()` interface as a standard `transformers` model. This is why models exported outside Optimum's convention (like LFM2) don't work here — the KV-cache tensor naming doesn't match what Optimum expects.

**fp32 vs fp16 vs int8 variants**
Most ONNX repos ship multiple precision variants. fp32 is the full-precision baseline. fp16 halves memory usage with negligible quality loss. int8 quantizes weights to 8-bit integers — roughly 4× smaller than fp32, with a small quality tradeoff. `from_pretrained` picks the default (usually fp32); pass `file_name="onnx/model_int8.onnx"` to force a specific one.

**Chat templates**
Instruction-tuned models expect their input formatted in a specific way (e.g. `<|user|>` tags, `[INST]` markers). The tokenizer's `chat_template` field encodes this. Models without one fall back to a plain `User: / Assistant:` prompt, which works but may produce lower-quality responses for models that were fine-tuned on a specific format.